In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split.zip

Streaming output truncated to the last 5000 lines.
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_50_1_box9.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam2_67_1_box24.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam4_52_1_box6.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_83_1_box26.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_222_1_box15.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_47_1_box0.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_24_1_box40.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_128_1_box29.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_88_1_box36.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_190_1_box47.jpg  
  inflating: content/OMR_5Fold_ROIs_split/Fold_2/val/confirmed/exam5_57_1_box19.jpg  
  i

In [7]:
import os
len(os.listdir("/content/content/OMR_5Fold_ROIs_split/Fold_1/train_Scen2_withGAN/crossedout"))

2500

In [4]:
import os
import shutil
import glob

K_FOLDS_DIR = "/content/content/OMR_5Fold_ROIs_split"
GAN_IMG_DIR = "/content/drive/MyDrive/OMR-Datasets/gan/gan_images_best"

for fold in range(1, 6):
    print(f"\n{'='*60}")
    print(f"🚀 ĐANG TỔ CHỨC DỮ LIỆU FOLD {fold}")

    fold_dir = os.path.join(K_FOLDS_DIR, f"Fold_{fold}")

    # Định nghĩa các đường dẫn đích
    train_old_dir = os.path.join(fold_dir, "train")
    scen1_dir = os.path.join(fold_dir, "train_Scen1_withoutGAN")
    scen2_dir = os.path.join(fold_dir, "train_Scen2_withGAN")

    # ==========================================
    # ⚠️ BẠN HÃY SỬA ĐƯỜNG DẪN NÀY CHO KHỚP THỰC TẾ
    # Giả sử bạn lưu ảnh GAN sinh ra của Fold 1 ở Fold_1/gan_images
    gan_source_dir = os.path.join(GAN_IMG_DIR, f"Fold_{fold}")
    # ==========================================

    # 1. ĐỔI TÊN THƯ MỤC TRAIN GỐC THÀNH SCEN 1
    if os.path.exists(train_old_dir):
        os.rename(train_old_dir, scen1_dir)
        print(f"   ✅ Đã đổi tên: 'train' -> 'train_Scen1_withoutGAN'")
    elif os.path.exists(scen1_dir):
        print(f"   ✅ Thư mục 'train_Scen1_withoutGAN' đã tồn tại sẵn.")
    else:
        print(f"   ❌ Cảnh báo: Không tìm thấy thư mục train gốc!")

    # 2. TẠO SCEN 2 VÀ COPY TOÀN BỘ TỪ SCEN 1 SANG
    print(f"   ⏳ Đang nhân bản (copy) toàn bộ ảnh từ Scen1 sang Scen2...")
    if os.path.exists(scen1_dir):
        # copytree với dirs_exist_ok=True sẽ tự động tạo thư mục Scen2 nếu chưa có
        shutil.copytree(scen1_dir, scen2_dir, dirs_exist_ok=True)
        print(f"   ✅ Đã tạo Scen2 và copy thành công.")

    # 3. BƠM 1000 ẢNH GAN VÀO LỚP CROSSEDOUT CỦA SCEN 2
    scen2_crossedout_dir = os.path.join(scen2_dir, "crossedout")

    if os.path.exists(gan_source_dir):
        gan_images = glob.glob(os.path.join(gan_source_dir, "*.*"))
        print(f"   ⏳ Đang copy {len(gan_images)} ảnh GAN vào Scen2/crossedout...")

        # Copy từng ảnh GAN đè vào mục crossedout
        for img_path in gan_images:
            shutil.copy(img_path, scen2_crossedout_dir)

        print("   ✅ Đã trộn ảnh GAN thành công.")
    else:
        print(f"   ❌ Cảnh báo: Không tìm thấy thư mục ảnh GAN tại: {gan_source_dir}")
        print("      (Vui lòng kiểm tra lại biến 'gan_source_dir' ở dòng 17)")

    # 4. THỐNG KÊ KIỂM TRA SỐ LƯỢNG (CHECK TRÙNG LẶP/THIẾU SÓT)
    print(f"   📊 Thống kê thư mục [train_Scen2_withGAN]:")
    for cls_name in ['confirmed', 'empty', 'crossedout']:
        cls_path = os.path.join(scen2_dir, cls_name)
        if os.path.exists(cls_path):
            num_imgs = len(os.listdir(cls_path))
            print(f"      - Lớp '{cls_name}': {num_imgs} ảnh")
        else:
            print(f"      - Lớp '{cls_name}': 0 ảnh (Không tìm thấy!)")


🚀 ĐANG TỔ CHỨC DỮ LIỆU FOLD 1
   ✅ Đã đổi tên: 'train' -> 'train_Scen1_withoutGAN'
   ⏳ Đang nhân bản (copy) toàn bộ ảnh từ Scen1 sang Scen2...
   ✅ Đã tạo Scen2 và copy thành công.
   ⏳ Đang copy 1000 ảnh GAN vào Scen2/crossedout...
   ✅ Đã trộn ảnh GAN thành công.
   📊 Thống kê thư mục [train_Scen2_withGAN]:
      - Lớp 'confirmed': 6605 ảnh
      - Lớp 'empty': 13484 ảnh
      - Lớp 'crossedout': 2500 ảnh

🚀 ĐANG TỔ CHỨC DỮ LIỆU FOLD 2
   ✅ Đã đổi tên: 'train' -> 'train_Scen1_withoutGAN'
   ⏳ Đang nhân bản (copy) toàn bộ ảnh từ Scen1 sang Scen2...
   ✅ Đã tạo Scen2 và copy thành công.
   ⏳ Đang copy 1000 ảnh GAN vào Scen2/crossedout...
   ✅ Đã trộn ảnh GAN thành công.
   📊 Thống kê thư mục [train_Scen2_withGAN]:
      - Lớp 'confirmed': 6555 ảnh
      - Lớp 'empty': 13361 ảnh
      - Lớp 'crossedout': 2500 ảnh

🚀 ĐANG TỔ CHỨC DỮ LIỆU FOLD 3
   ✅ Đã đổi tên: 'train' -> 'train_Scen1_withoutGAN'
   ⏳ Đang nhân bản (copy) toàn bộ ảnh từ Scen1 sang Scen2...
   ✅ Đã tạo Scen2 và copy thà

In [8]:
source = K_FOLDS_DIR
destination = '/content'
!mv {source} {destination}

In [10]:
import os

# Define the directory to be zipped (K_FOLDS_DIR is defined in the previous cell)
zip_source_dir = '/content/OMR_5Fold_ROIs_split'

# Define the destination path in Google Drive
# Get the base name of the directory to use as the zip file name
zip_filename = os.path.basename(zip_source_dir) + '_v4.zip'
zip_destination_path = os.path.join('/content/drive/MyDrive/OMR-Datasets', zip_filename)

print(f"Zipping '{zip_source_dir}' to '{zip_destination_path}'...")

# Use !zip command to zip the directory
# -r for recursive, -q for quiet (optional)
!zip -r -q {zip_destination_path} {zip_source_dir}

print(f"Successfully zipped '{zip_source_dir}' to '{zip_destination_path}'")

Zipping '/content/OMR_5Fold_ROIs_split' to '/content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split_v4.zip'...
Successfully zipped '/content/OMR_5Fold_ROIs_split' to '/content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split_v4.zip'
